# Module 04 — ML Theory & Evaluation
## 04-05: Handling Imbalanced Data

**Objective:** Build a complete imbalanced-data toolkit from scratch — random oversampling, SMOTE, random undersampling, class-weighted losses, and threshold tuning — then apply and compare all strategies on a synthetic imbalanced classification problem.

**Prerequisites:** 4-01 (Evaluation Metrics), 4-02 (Cross-Validation)

## Part 0 — Setup & Prerequisites

Class imbalance is one of the most pervasive real-world ML challenges: fraud detection datasets may
have 0.1% positive examples; medical diagnoses 1–5%; anomaly detection even less.  Standard accuracy
is **misleading** under imbalance — a model that always predicts the majority class achieves 99%
accuracy on a 1:99 dataset while detecting exactly zero positives.  This notebook implements and
compares the four main remedies:

1. **Resampling** — oversample the minority or undersample the majority
2. **Synthetic oversampling** (SMOTE) — generate new minority examples via interpolation
3. **Class-weighted loss** — penalise misclassifying the minority class more heavily
4. **Threshold tuning** — shift the decision boundary away from the default 0.5

> **Note:** Evaluation metrics, PR curves, and stratified splitting are taught in 4-01 and 4-02
> respectively — we use them here without re-deriving them.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
import time
import random
import warnings
from typing import Any
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    precision_recall_curve,
    roc_curve,
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

from sklearn.decomposition import PCA
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
N_SAMPLES    = 5000
N_FEATURES   = 20
MINORITY_FRAC = 0.05    # 5 % positive class → 1:19 imbalance ratio
TEST_SIZE    = 0.2
N_FOLDS      = 5

### Data Generation & Exploration

We generate a binary classification problem with a **1:19 imbalance ratio** (5% positives).
`make_classification` lets us control the number of informative, redundant, and noisy features,
giving us a realistic tabular dataset where the class boundary is learnable but non-trivial.

In [ ]:
# ── Generate imbalanced synthetic dataset ─────────────────────────────────────
weights = [1.0 - MINORITY_FRAC, MINORITY_FRAC]   # [majority, minority]

X_full, y_full = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=8,
    n_redundant=4,
    n_clusters_per_class=2,
    weights=weights,
    flip_y=0.01,          # 1% label noise — realistic setting
    random_state=SEED,
)

# Stratified 80/20 train/test split — see 4-02 for stratified splitting
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=TEST_SIZE, random_state=SEED, stratify=y_full
)

# Normalise features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

n_majority = int((y_train == 0).sum())
n_minority = int((y_train == 1).sum())
imbalance_ratio = n_majority / n_minority

print(f"Training set: {len(y_train):,} samples")
print(f"  Majority (class 0): {n_majority:,}  ({n_majority/len(y_train):.1%})")
print(f"  Minority (class 1): {n_minority:,}  ({n_minority/len(y_train):.1%})")
print(f"  Imbalance ratio   : {imbalance_ratio:.1f}:1")
print(f"\nTest set : {len(y_test):,} samples")
print(f"  Minority (class 1): {int((y_test==1).sum()):,}  ({(y_test==1).mean():.1%})")

In [ ]:
# ── EDA: class distribution and naive baseline ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Bar chart of class distribution
class_counts = np.bincount(y_train)
axes[0].bar(["Majority (0)", "Minority (1)"], class_counts,
            color=["steelblue", "coral"], edgecolor="white", width=0.5)
for i, cnt in enumerate(class_counts):
    axes[0].text(i, cnt + 30, str(cnt), ha="center", fontweight="bold")
axes[0].set_title("Training Class Distribution", fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].grid(True, axis="y", alpha=0.3)

# 2-D PCA projection to visualise overlap
pca = PCA(n_components=2, random_state=SEED)
X_2d = pca.fit_transform(X_train)
for label, color, name in [(0, "steelblue", "Majority"), (1, "coral", "Minority")]:
    mask = y_train == label
    axes[1].scatter(X_2d[mask, 0], X_2d[mask, 1],
                    c=color, s=8, alpha=0.4, label=name)
axes[1].set_title("PCA Projection (2D)", fontweight="bold")
axes[1].legend()
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

# Naive majority-class baseline metrics
y_naive = np.zeros(len(y_test), dtype=int)   # always predict majority
naive_acc  = (y_naive == y_test).mean()
naive_f1   = f1_score(y_test, y_naive, zero_division=0)
naive_prec = (y_naive[y_test == 1] == 1).sum() / max(1, (y_naive == 1).sum())
naive_rec  = (y_naive[y_test == 1] == 1).sum() / max(1, (y_test == 1).sum())

metrics_names = ["Accuracy", "F1 (minority)", "Precision", "Recall"]
metrics_vals  = [naive_acc, naive_f1, float(naive_prec), float(naive_rec)]
axes[2].barh(metrics_names, metrics_vals, color="steelblue", edgecolor="white")
axes[2].axvline(0.5, color="red", linestyle="--", alpha=0.6, label="50% reference")
axes[2].set_title("Naive Majority-Class Baseline", fontweight="bold")
axes[2].set_xlabel("Score")
axes[2].set_xlim(0, 1.05)
axes[2].legend()
for i, v in enumerate(metrics_vals):
    axes[2].text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=9)

plt.suptitle("Imbalanced Dataset EDA — 5% Minority Class",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Naive baseline accuracy: {naive_acc:.3f}  — misleadingly high!")
print(f"Naive baseline F1      : {naive_f1:.3f}  — reveals the true failure.")

---
## Part 1 — Imbalanced-Data Techniques from Scratch

### 1.1 Random Oversampling

The simplest resampling strategy: **duplicate minority samples** with replacement until the minority
class matches a target ratio.  Cheap and effective, but it can cause the model to memorise specific
minority examples rather than learning the decision boundary.

$$N_{\text{minority}}^{\text{new}} = \left\lfloor \frac{N_{\text{majority}}}{\text{target ratio}} \right\rfloor$$

In [ ]:
# ── Random Oversampling ───────────────────────────────────────────────────────

def random_oversample(
    X: np.ndarray,
    y: np.ndarray,
    target_ratio: float = 1.0,
    seed: int = SEED,
) -> tuple[np.ndarray, np.ndarray]:
    '''Oversample the minority class by duplicating examples with replacement.

    Samples minority rows until the minority-to-majority count ratio equals
    target_ratio.  Does not modify majority class samples.

    Args:
        X: Feature matrix of shape (n_samples, n_features).
        y: Binary label vector of shape (n_samples,).
        target_ratio: Desired minority/majority ratio after resampling.
            1.0 = balanced (1:1), 0.5 = 1:2 ratio, etc.
        seed: Random seed for reproducibility.

    Returns:
        Tuple of (X_resampled, y_resampled) with augmented minority class.
    '''
    rng = np.random.default_rng(seed)
    classes, counts = np.unique(y, return_counts=True)
    majority_class = classes[np.argmax(counts)]
    minority_class = classes[np.argmin(counts)]

    n_majority = int(counts.max())
    n_minority_target = int(n_majority * target_ratio)
    n_minority_current = int(counts.min())
    n_to_add = max(0, n_minority_target - n_minority_current)

    minority_idx = np.where(y == minority_class)[0]
    sampled_idx = rng.choice(minority_idx, size=n_to_add, replace=True)

    X_resampled = np.vstack([X, X[sampled_idx]])
    y_resampled = np.concatenate([y, y[sampled_idx]])

    # Shuffle to prevent ordering artefacts in batch-based training
    shuffle_idx = rng.permutation(len(y_resampled))
    return X_resampled[shuffle_idx], y_resampled[shuffle_idx]


# Validate
X_over, y_over = random_oversample(X_train, y_train, target_ratio=1.0)
over_counts = np.bincount(y_over)
print(f"After random oversampling (target 1:1):")
print(f"  Majority: {over_counts[0]:,}  Minority: {over_counts[1]:,}")
print(f"  Ratio   : {over_counts[0]/over_counts[1]:.2f}:1")
assert over_counts[0] == n_majority, "Majority class should be unchanged"
print("Random oversampling validated.")

### 1.2 SMOTE — Synthetic Minority Oversampling Technique

SMOTE (Chawla et al., 2002) generates **synthetic** minority examples by interpolating between
existing minority samples and their k-nearest minority neighbours:

$$\tilde{x} = x_i + \lambda \cdot (x_{\text{nn}} - x_i), \quad \lambda \sim \mathcal{U}(0, 1)$$

This prevents the memorisation problem of random oversampling by creating novel points in the
feature space along minority-class manifolds.  The key hyperparameter is $k$ (number of neighbours);
larger $k$ produces smoother synthetic distributions.

In [ ]:
# ── SMOTE ─────────────────────────────────────────────────────────────────────

def _compute_knn_indices(
    X_source: np.ndarray,
    X_query: np.ndarray,
    k: int,
) -> np.ndarray:
    '''Return indices of k nearest neighbours in X_source for each row of X_query.

    Uses Euclidean distance with a vectorised pairwise distance computation.
    For small minority sets (< 5000 rows) this is fast enough without an
    approximate-nearest-neighbour index.

    Args:
        X_source: Candidate neighbour points of shape (n_source, n_features).
        X_query: Query points of shape (n_query, n_features).
        k: Number of nearest neighbours to return.

    Returns:
        Integer index array of shape (n_query, k).
    '''
    # Pairwise squared Euclidean distances: (n_query, n_source)
    diff = X_query[:, np.newaxis, :] - X_source[np.newaxis, :, :]   # (q, s, d)
    dist_sq = (diff ** 2).sum(axis=2)                                # (q, s)
    # Exclude self (distance 0) by setting diagonal to inf when source == query
    nn_idx = np.argsort(dist_sq, axis=1)[:, 1: k + 1]               # skip rank-0
    return nn_idx


def smote_oversample(
    X: np.ndarray,
    y: np.ndarray,
    target_ratio: float = 1.0,
    k_neighbours: int = 5,
    seed: int = SEED,
) -> tuple[np.ndarray, np.ndarray]:
    '''Apply SMOTE to generate synthetic minority-class examples.

    For each synthetic sample needed, picks a random minority example x_i
    and one of its k nearest minority neighbours x_nn, then interpolates:
    x_new = x_i + lambda * (x_nn - x_i)  where lambda ~ Uniform(0, 1).

    Args:
        X: Feature matrix of shape (n_samples, n_features).
        y: Binary label vector of shape (n_samples,).
        target_ratio: Desired minority/majority ratio after synthesis.
        k_neighbours: Number of nearest minority neighbours to consider.
        seed: Random seed.

    Returns:
        Tuple of (X_resampled, y_resampled) with synthetic minority examples.
    '''
    rng = np.random.default_rng(seed)
    classes, counts = np.unique(y, return_counts=True)
    minority_class = classes[np.argmin(counts)]

    n_majority = int(counts.max())
    n_minority_current = int(counts.min())
    n_minority_target  = int(n_majority * target_ratio)
    n_to_synthesise    = max(0, n_minority_target - n_minority_current)

    if n_to_synthesise == 0:
        return X.copy(), y.copy()

    X_minority = X[y == minority_class]
    k_actual = min(k_neighbours, len(X_minority) - 1)

    # Pre-compute k-NN within the minority set
    nn_indices = _compute_knn_indices(X_minority, X_minority, k_actual)  # (m, k)

    # Generate synthetic samples
    synthetic_X = np.zeros((n_to_synthesise, X.shape[1]), dtype=X.dtype)
    anchor_indices = rng.integers(0, len(X_minority), size=n_to_synthesise)

    for idx, anchor_i in enumerate(anchor_indices):
        nn_choice = rng.integers(0, k_actual)
        neighbour_i = nn_indices[anchor_i, nn_choice]
        lam = rng.uniform(0.0, 1.0)
        synthetic_X[idx] = (
            X_minority[anchor_i] + lam * (X_minority[neighbour_i] - X_minority[anchor_i])
        )

    synthetic_y = np.full(n_to_synthesise, minority_class, dtype=y.dtype)

    X_resampled = np.vstack([X, synthetic_X])
    y_resampled = np.concatenate([y, synthetic_y])

    shuffle_idx = rng.permutation(len(y_resampled))
    return X_resampled[shuffle_idx], y_resampled[shuffle_idx]


# Validate SMOTE
X_smote, y_smote = smote_oversample(X_train, y_train, target_ratio=1.0, k_neighbours=5)
smote_counts = np.bincount(y_smote)
print(f"After SMOTE (target 1:1):")
print(f"  Majority: {smote_counts[0]:,}  Minority: {smote_counts[1]:,}")
print(f"  New samples generated: {smote_counts[1] - n_minority:,}")

# Verify synthetic samples lie between existing minority samples
X_minority_orig = X_train[y_train == 1]
X_synthetic = X_smote[len(X_train):][y_smote[len(X_train):] == 1]
# Each synthetic sample should be within the convex hull of minority class
# Rough check: synthetic feature range should not exceed original minority range
orig_min = X_minority_orig.min(axis=0)
orig_max = X_minority_orig.max(axis=0)
within_range = np.all(
    (X_synthetic >= orig_min - 1e-8) & (X_synthetic <= orig_max + 1e-8)
)
print(f"  All synthetic samples within original minority range: {within_range}")
print("SMOTE validated.")

### 1.3 Random Undersampling

The complementary approach: **discard majority samples** until the imbalance ratio reaches the target.
Faster to train (smaller dataset) but risks discarding potentially informative majority examples.
Best combined with an ensemble (BalancedBagging) to reduce information loss.

In [ ]:
# ── Random Undersampling ──────────────────────────────────────────────────────

def random_undersample(
    X: np.ndarray,
    y: np.ndarray,
    target_ratio: float = 1.0,
    seed: int = SEED,
) -> tuple[np.ndarray, np.ndarray]:
    '''Undersample the majority class by removing examples at random.

    Reduces the majority class size until the minority/majority ratio equals
    target_ratio.  Does not modify minority class samples.

    Args:
        X: Feature matrix of shape (n_samples, n_features).
        y: Binary label vector of shape (n_samples,).
        target_ratio: Desired minority/majority ratio after undersampling.
            1.0 = balanced, 0.5 = keep twice as many majority examples.
        seed: Random seed.

    Returns:
        Tuple of (X_resampled, y_resampled) with reduced majority class.
    '''
    rng = np.random.default_rng(seed)
    classes, counts = np.unique(y, return_counts=True)
    majority_class = classes[np.argmax(counts)]
    minority_class = classes[np.argmin(counts)]

    n_minority = int(counts.min())
    n_majority_target = int(n_minority / target_ratio)

    majority_idx = np.where(y == majority_class)[0]
    minority_idx = np.where(y == minority_class)[0]

    keep_majority = rng.choice(majority_idx, size=n_majority_target, replace=False)
    selected_idx  = np.concatenate([keep_majority, minority_idx])

    shuffle_idx = rng.permutation(len(selected_idx))
    final_idx   = selected_idx[shuffle_idx]
    return X[final_idx], y[final_idx]


# Validate
X_under, y_under = random_undersample(X_train, y_train, target_ratio=1.0)
under_counts = np.bincount(y_under)
print(f"After random undersampling (target 1:1):")
print(f"  Majority: {under_counts[0]:,}  Minority: {under_counts[1]:,}")
print(f"  Dataset reduced from {len(y_train):,} → {len(y_under):,} samples")
print(f"  Information retained: {len(y_under)/len(y_train):.1%}")
print("Random undersampling validated.")

### 1.4 Class-Weighted Loss

Instead of resampling, we can **re-weight the loss function** so that misclassifying a minority
example incurs a higher penalty.  The standard inverse-frequency weighting is:

$$w_c = \frac{N}{K \cdot N_c}$$

where $N$ is total samples, $K$ is number of classes, and $N_c$ is samples in class $c$.
For a 1:19 imbalanced dataset this gives minority weight ≈ 9.5× the majority weight.

In [ ]:
# ── Class Weight Computation ───────────────────────────────────────────────────

def compute_class_weights(
    y: np.ndarray,
    strategy: str = "inverse_frequency",
) -> dict[int, float]:
    '''Compute per-class loss weights to counteract class imbalance.

    Two strategies are supported:

    - ``"inverse_frequency"``: w_c = N / (K * N_c).  Standard sklearn convention.
    - ``"sqrt_inverse"``: w_c = sqrt(N / N_c).  More conservative; reduces
      the range of weights for extreme imbalances.

    Args:
        y: Label vector of shape (n_samples,).
        strategy: Weighting strategy; one of ``"inverse_frequency"`` or
            ``"sqrt_inverse"``.

    Returns:
        Dictionary mapping class index to its loss weight.
    '''
    classes, counts = np.unique(y, return_counts=True)
    n_samples = len(y)
    n_classes  = len(classes)

    weights: dict[int, float] = {}
    for cls, cnt in zip(classes, counts):
        if strategy == "inverse_frequency":
            weights[int(cls)] = float(n_samples / (n_classes * cnt))
        elif strategy == "sqrt_inverse":
            weights[int(cls)] = float(np.sqrt(n_samples / cnt))
        else:
            raise ValueError(f"Unknown strategy: {strategy!r}")

    return weights


weights_inv = compute_class_weights(y_train, strategy="inverse_frequency")
weights_sqrt = compute_class_weights(y_train, strategy="sqrt_inverse")

print("Class weights (inverse_frequency):")
for cls, w in weights_inv.items():
    print(f"  Class {cls}: {w:.4f}")
print(f"  Weight ratio minority/majority: {weights_inv[1]/weights_inv[0]:.2f}x")

print("\nClass weights (sqrt_inverse):")
for cls, w in weights_sqrt.items():
    print(f"  Class {cls}: {w:.4f}")
print(f"  Weight ratio minority/majority: {weights_sqrt[1]/weights_sqrt[0]:.2f}x")

### 1.5 Threshold Tuning

Even with a well-calibrated model, the **default decision threshold of 0.5 is rarely optimal** for
imbalanced data.  By sweeping the threshold and inspecting the precision-recall trade-off, we can
find the operating point that maximises $F_1$ (or any $F_\beta$ metric for other precision/recall preferences).

$$F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

In [ ]:
# ── Threshold Tuning ──────────────────────────────────────────────────────────

def find_optimal_threshold(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    beta: float = 1.0,
    n_thresholds: int = 200,
) -> tuple[float, float, float, float]:
    '''Find the decision threshold that maximises F_beta score.

    Sweeps thresholds uniformly over [0.01, 0.99] and computes F_beta at
    each point.  Returns the threshold, precision, recall, and F_beta at the
    optimal operating point.

    Args:
        y_true: Ground-truth binary labels of shape (n_samples,).
        y_proba: Predicted probabilities for the positive class.
        beta: F_beta parameter.  beta=1 maximises F1; beta=2 up-weights recall;
            beta=0.5 up-weights precision.
        n_thresholds: Number of threshold candidates to evaluate.

    Returns:
        Tuple of (best_threshold, precision_at_best, recall_at_best, fbeta_at_best).
    '''
    thresholds = np.linspace(0.01, 0.99, n_thresholds)
    best_fbeta = -1.0
    best_thresh = 0.5
    best_prec = 0.0
    best_rec  = 0.0

    for thresh in thresholds:
        y_pred = (y_proba >= thresh).astype(int)
        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        denom = (beta ** 2) * prec + rec
        fbeta = (1 + beta ** 2) * prec * rec / denom if denom > 0 else 0.0

        if fbeta > best_fbeta:
            best_fbeta = fbeta
            best_thresh = thresh
            best_prec  = prec
            best_rec   = rec

    return best_thresh, best_prec, best_rec, best_fbeta


def threshold_sweep(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    n_thresholds: int = 200,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    '''Compute precision, recall, and F1 across a range of thresholds.

    Args:
        y_true: Ground-truth binary labels.
        y_proba: Predicted positive-class probabilities.
        n_thresholds: Number of evenly-spaced threshold values to evaluate.

    Returns:
        Tuple of (thresholds, precisions, recalls, f1_scores) arrays.
    '''
    thresholds = np.linspace(0.01, 0.99, n_thresholds)
    precisions = np.zeros(n_thresholds)
    recalls    = np.zeros(n_thresholds)
    f1_scores  = np.zeros(n_thresholds)

    for i, thresh in enumerate(thresholds):
        y_pred = (y_proba >= thresh).astype(int)
        tp = int(((y_pred == 1) & (y_true == 1)).sum())
        fp = int(((y_pred == 1) & (y_true == 0)).sum())
        fn = int(((y_pred == 0) & (y_true == 1)).sum())

        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

        precisions[i] = prec
        recalls[i]    = rec
        f1_scores[i]  = f1

    return thresholds, precisions, recalls, f1_scores


print("Threshold tuning functions defined.")

---
## Part 2 — ImbalancedPipeline: Assembling the Toolkit

We wrap all resampling strategies into an `ImbalancedPipeline` class that provides a unified
`fit_resample()` interface, and implement stratified K-fold cross-validation adapted for imbalanced
settings — where stratification is **essential** to maintain minority class representation in each fold.

In [ ]:
# ── ImbalancedPipeline ────────────────────────────────────────────────────────

class ImbalancedPipeline:
    '''Unified interface for imbalanced-data resampling strategies.

    Wraps random oversampling, SMOTE, and random undersampling under a single
    ``fit_resample()`` API, following the imbalanced-learn convention.

    Attributes:
        strategy: Resampling strategy name.
        target_ratio: Desired minority/majority ratio after resampling.
        k_neighbours: k for SMOTE (ignored for other strategies).
        seed: Random seed.
    '''

    VALID_STRATEGIES = ("oversample", "smote", "undersample", "none")

    def __init__(
        self,
        strategy: str = "smote",
        target_ratio: float = 1.0,
        k_neighbours: int = 5,
        seed: int = SEED,
    ) -> None:
        '''Initialise the imbalanced-data pipeline.

        Args:
            strategy: One of ``"oversample"``, ``"smote"``, ``"undersample"``,
                or ``"none"`` (no resampling, used as baseline).
            target_ratio: Desired minority/majority ratio after resampling.
                1.0 = fully balanced.
            k_neighbours: Number of nearest minority neighbours for SMOTE.
            seed: Random seed for reproducibility.
        '''
        if strategy not in self.VALID_STRATEGIES:
            raise ValueError(
                f"strategy must be one of {self.VALID_STRATEGIES}, got {strategy!r}"
            )
        self.strategy      = strategy
        self.target_ratio  = target_ratio
        self.k_neighbours  = k_neighbours
        self.seed          = seed

    def fit_resample(
        self,
        X: np.ndarray,
        y: np.ndarray,
    ) -> tuple[np.ndarray, np.ndarray]:
        '''Apply the configured resampling strategy and return balanced arrays.

        Args:
            X: Feature matrix of shape (n_samples, n_features).
            y: Label vector of shape (n_samples,).

        Returns:
            Tuple of (X_resampled, y_resampled).
        '''
        if self.strategy == "oversample":
            return random_oversample(X, y, self.target_ratio, self.seed)
        elif self.strategy == "smote":
            return smote_oversample(X, y, self.target_ratio, self.k_neighbours, self.seed)
        elif self.strategy == "undersample":
            return random_undersample(X, y, self.target_ratio, self.seed)
        else:  # "none"
            return X.copy(), y.copy()

    def __repr__(self) -> str:
        '''Return string representation.

        Returns:
            Descriptive string.
        '''
        return (
            f"ImbalancedPipeline(strategy={self.strategy!r}, "
            f"target_ratio={self.target_ratio})"
        )


# ── Stratified K-Fold CV for imbalanced data ──────────────────────────────────

def imbalanced_stratified_cv(
    estimator: Any,
    X: np.ndarray,
    y: np.ndarray,
    pipeline: ImbalancedPipeline,
    n_splits: int = 5,
    class_weight: dict | None = None,
    seed: int = SEED,
) -> dict[str, list]:
    '''Run stratified K-fold CV with resampling applied inside each fold.

    Resampling is fit **only on the training fold** of each split, preventing
    data leakage from the validation fold into the synthetic samples.

    Args:
        estimator: sklearn-compatible classifier with fit/predict_proba.
        X: Feature matrix of shape (n_samples, n_features).
        y: Label vector of shape (n_samples,).
        pipeline: ImbalancedPipeline instance to apply per fold.
        n_splits: Number of CV folds.
        class_weight: Optional per-class weight dict passed to ``estimator.fit``
            via the ``sample_weight`` parameter.
        seed: Random seed for the KFold splitter.

    Returns:
        Dict with keys ``"f1"``, ``"roc_auc"``, ``"avg_precision"`` containing
        per-fold metric lists.
    '''
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    results: dict[str, list] = {"f1": [], "roc_auc": [], "avg_precision": []}

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X[train_idx], y[train_idx]
        X_val, y_val = X[val_idx], y[val_idx]

        # Resample only the training fold — never touch val fold
        fold_pipe = ImbalancedPipeline(
            strategy=pipeline.strategy,
            target_ratio=pipeline.target_ratio,
            k_neighbours=pipeline.k_neighbours,
            seed=seed + fold_idx,   # different seed per fold
        )
        X_tr_res, y_tr_res = fold_pipe.fit_resample(X_tr, y_tr)

        # Build sample_weight if class_weight is provided
        if class_weight is not None:
            sample_weight = np.array([class_weight[int(c)] for c in y_tr_res])
            estimator.fit(X_tr_res, y_tr_res, sample_weight=sample_weight)
        else:
            estimator.fit(X_tr_res, y_tr_res)

        y_proba = estimator.predict_proba(X_val)[:, 1]
        y_pred  = (y_proba >= 0.5).astype(int)

        results["f1"].append(f1_score(y_val, y_pred, zero_division=0))
        results["roc_auc"].append(roc_auc_score(y_val, y_proba))
        results["avg_precision"].append(average_precision_score(y_val, y_proba))

    return results


print("ImbalancedPipeline and imbalanced_stratified_cv defined.")

---
## Part 3 — Applying All Strategies to the Imbalanced Dataset

We train a Logistic Regression classifier under six conditions and evaluate on the held-out test set:

| Config | Resampling | Class weight | Threshold |
|--------|-----------|--------------|----------|
| Baseline | None | None | 0.5 |
| Class-weighted | None | inverse_freq | 0.5 |
| Oversampled | Random | None | 0.5 |
| Undersampled | Random | None | 0.5 |
| SMOTE | SMOTE | None | 0.5 |
| SMOTE + Threshold | SMOTE | None | optimal F1 |

In [ ]:
# ── Train and evaluate all six configurations ─────────────────────────────────

def evaluate_config(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_te: np.ndarray,
    y_te: np.ndarray,
    sample_weight: np.ndarray | None = None,
    threshold: float = 0.5,
) -> dict[str, float]:
    '''Fit LogisticRegression and evaluate on test set at a given threshold.

    Args:
        X_tr: Training feature matrix.
        y_tr: Training labels.
        X_te: Test feature matrix.
        y_te: Test labels.
        sample_weight: Optional per-sample loss weights for training.
        threshold: Decision threshold for converting probabilities to labels.

    Returns:
        Dict of metric name → scalar value.
    '''
    model = LogisticRegression(max_iter=1000, random_state=SEED)
    if sample_weight is not None:
        model.fit(X_tr, y_tr, sample_weight=sample_weight)
    else:
        model.fit(X_tr, y_tr)

    y_proba = model.predict_proba(X_te)[:, 1]
    y_pred  = (y_proba >= threshold).astype(int)

    tp = int(((y_pred == 1) & (y_te == 1)).sum())
    fp = int(((y_pred == 1) & (y_te == 0)).sum())
    fn = int(((y_pred == 0) & (y_te == 1)).sum())
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

    return {
        "f1"            : f1,
        "precision"     : prec,
        "recall"        : rec,
        "roc_auc"       : roc_auc_score(y_te, y_proba),
        "avg_precision" : average_precision_score(y_te, y_proba),
        "threshold"     : threshold,
        "y_proba"       : y_proba,   # stored for PR/ROC curve plots
    }


# Build all resampled datasets
pipe_none  = ImbalancedPipeline(strategy="none")
pipe_over  = ImbalancedPipeline(strategy="oversample")
pipe_under = ImbalancedPipeline(strategy="undersample")
pipe_smote = ImbalancedPipeline(strategy="smote", k_neighbours=5)

X_none,  y_none  = pipe_none.fit_resample(X_train, y_train)
X_over,  y_over  = pipe_over.fit_resample(X_train, y_train)
X_under, y_under = pipe_under.fit_resample(X_train, y_train)
X_smote, y_smote = pipe_smote.fit_resample(X_train, y_train)

# Class weights for the weighted config (fit on original training data)
cw = compute_class_weights(y_train, strategy="inverse_frequency")
sample_weights_none = np.array([cw[int(c)] for c in y_none])

# Train Logistic Regression for each config, find optimal threshold on training proba
t0 = time.time()

# --- Baseline ---
res_baseline = evaluate_config(X_none, y_none, X_test, y_test)

# --- Class-weighted ---
res_weighted = evaluate_config(X_none, y_none, X_test, y_test,
                               sample_weight=sample_weights_none)

# --- Random oversampled ---
res_over = evaluate_config(X_over, y_over, X_test, y_test)

# --- Random undersampled ---
res_under = evaluate_config(X_under, y_under, X_test, y_test)

# --- SMOTE ---
res_smote = evaluate_config(X_smote, y_smote, X_test, y_test)

# --- SMOTE + optimal threshold (find threshold on val predictions from training)
# We use the test set for threshold search here for illustration — in practice,
# use a held-out validation set to avoid leakage
best_thresh, best_p, best_r, best_f1 = find_optimal_threshold(
    y_test, res_smote["y_proba"], beta=1.0
)
res_smote_thresh = evaluate_config(
    X_smote, y_smote, X_test, y_test, threshold=best_thresh
)

elapsed = time.time() - t0
print(f"All 6 configs trained and evaluated in {elapsed:.1f}s.")
print(f"Optimal threshold for SMOTE: {best_thresh:.3f}")

In [ ]:
# ── Stratified 5-fold CV comparison (inside-fold resampling) ──────────────────
base_estimator = LogisticRegression(max_iter=1000, random_state=SEED)

cv_configs: dict = {
    "Baseline":       (ImbalancedPipeline(strategy="none"),   None),
    "Class-weighted": (ImbalancedPipeline(strategy="none"),   cw),
    "Oversampled":    (ImbalancedPipeline(strategy="oversample"), None),
    "Undersampled":   (ImbalancedPipeline(strategy="undersample"), None),
    "SMOTE":          (ImbalancedPipeline(strategy="smote"),   None),
}

cv_results: dict = {}
for config_name, (pipe, cw_dict) in cv_configs.items():
    cv_res = imbalanced_stratified_cv(
        LogisticRegression(max_iter=1000, random_state=SEED),
        X_train, y_train,
        pipeline=pipe,
        class_weight=cw_dict,
        n_splits=N_FOLDS,
    )
    cv_results[config_name] = cv_res
    print(
        f"{config_name:15s} | "
        f"F1: {np.mean(cv_res['f1']):.3f} ± {np.std(cv_res['f1']):.3f} | "
        f"ROC-AUC: {np.mean(cv_res['roc_auc']):.3f} | "
        f"AvgPrec: {np.mean(cv_res['avg_precision']):.3f}"
    )

print("\nNote: Resampling applied per fold (no leakage into validation folds).")

---
## Part 4 — Evaluation & Analysis

In [ ]:
# ── Summary results table ─────────────────────────────────────────────────────
all_results: dict = {
    "Baseline"      : res_baseline,
    "Class-weighted": res_weighted,
    "Oversampled"   : res_over,
    "Undersampled"  : res_under,
    "SMOTE"         : res_smote,
    "SMOTE+Thresh"  : res_smote_thresh,
}

rows = []
for config_name, metrics in all_results.items():
    rows.append({
        "Config"       : config_name,
        "Threshold"    : f"{metrics['threshold']:.3f}",
        "Precision"    : f"{metrics['precision']:.3f}",
        "Recall"       : f"{metrics['recall']:.3f}",
        "F1"           : f"{metrics['f1']:.3f}",
        "ROC-AUC"      : f"{metrics['roc_auc']:.3f}",
        "Avg Precision": f"{metrics['avg_precision']:.3f}",
    })

results_df = pd.DataFrame(rows)
print("Test-set performance by imbalanced-data strategy:")
print(results_df.to_string(index=False))

In [ ]:
# ── PR and ROC curves for all configs ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["steelblue", "coral", "green", "purple", "orange", "brown"]
linestyles = ["-", "-", "--", "--", "-.", ":"]

# Chance level for PR curve = minority fraction
chance_pr = (y_test == 1).mean()
axes[0].axhline(chance_pr, color="gray", linestyle=":", label=f"Chance ({chance_pr:.2f})")
axes[1].plot([0, 1], [0, 1], color="gray", linestyle=":", label="Chance")

for i, (config_name, metrics) in enumerate(all_results.items()):
    y_proba = metrics["y_proba"]
    color  = colors[i]
    ls     = linestyles[i]

    # PR curve
    prec_pts, rec_pts, _ = precision_recall_curve(y_test, y_proba)
    ap = metrics["avg_precision"]
    axes[0].plot(rec_pts, prec_pts, color=color, linestyle=ls,
                 label=f"{config_name} (AP={ap:.3f})")
    # Mark operating point
    axes[0].scatter([metrics["recall"]], [metrics["precision"]],
                    color=color, s=60, zorder=5)

    # ROC curve
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = metrics["roc_auc"]
    axes[1].plot(fpr, tpr, color=color, linestyle=ls,
                 label=f"{config_name} (AUC={auc:.3f})")

axes[0].set_xlabel("Recall", fontsize=11)
axes[0].set_ylabel("Precision", fontsize=11)
axes[0].set_title("Precision-Recall Curves\n(preferred for imbalanced data)",
                  fontsize=11, fontweight="bold")
axes[0].legend(fontsize=8, loc="upper right")
axes[0].set_xlim(0, 1.01)
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("False Positive Rate", fontsize=11)
axes[1].set_ylabel("True Positive Rate", fontsize=11)
axes[1].set_title("ROC Curves\n(less sensitive to imbalance ratio)",
                  fontsize=11, fontweight="bold")
axes[1].legend(fontsize=8, loc="lower right")
axes[1].grid(True, alpha=0.3)

plt.suptitle("PR vs ROC: Evaluating Imbalanced-Data Strategies",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Threshold tuning visualisation for SMOTE model ───────────────────────────
y_proba_smote = res_smote["y_proba"]
thresholds, precisions, recalls, f1_scores = threshold_sweep(y_test, y_proba_smote)

optimal_thresh, opt_prec, opt_rec, opt_f1 = find_optimal_threshold(
    y_test, y_proba_smote, beta=1.0
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision / Recall / F1 vs threshold
axes[0].plot(thresholds, precisions, label="Precision", color="steelblue", linewidth=2)
axes[0].plot(thresholds, recalls,    label="Recall",    color="coral",     linewidth=2)
axes[0].plot(thresholds, f1_scores,  label="F1",        color="green",     linewidth=2)
axes[0].axvline(0.5,            color="gray",  linestyle="--", alpha=0.7, label="Default (0.5)")
axes[0].axvline(optimal_thresh, color="black", linestyle="-",  alpha=0.9,
                label=f"Optimal ({optimal_thresh:.3f})")
axes[0].set_xlabel("Decision Threshold", fontsize=11)
axes[0].set_ylabel("Score", fontsize=11)
axes[0].set_title("Metrics vs Decision Threshold (SMOTE model)",
                  fontsize=11, fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# F1 at default vs optimal threshold
f1_default = res_smote["f1"]
f1_optimal = res_smote_thresh["f1"]
bar_labels  = [f"Default\n(t=0.50)\nF1={f1_default:.3f}",
               f"Optimal\n(t={optimal_thresh:.3f})\nF1={f1_optimal:.3f}"]
axes[1].bar(bar_labels, [f1_default, f1_optimal],
            color=["steelblue", "green"], edgecolor="white", width=0.4)
axes[1].set_ylabel("F1 Score (minority class)", fontsize=11)
axes[1].set_title("Default vs Optimal Threshold (SMOTE)",
                  fontsize=11, fontweight="bold")
axes[1].set_ylim(0, 1.0)
axes[1].grid(True, axis="y", alpha=0.3)
for i, val in enumerate([f1_default, f1_optimal]):
    axes[1].text(i, val + 0.01, f"{val:.3f}", ha="center", fontweight="bold", fontsize=11)

plt.suptitle("Threshold Tuning for Imbalanced Classification",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"F1 improvement from threshold tuning: "
      f"{f1_default:.3f} → {f1_optimal:.3f} "
      f"(+{f1_optimal - f1_default:+.3f})")

In [ ]:
# ── Confusion matrices: Baseline vs SMOTE+Threshold ───────────────────────────
configs_to_plot = {
    "Baseline (t=0.5)": res_baseline,
    "SMOTE+Threshold"  : res_smote_thresh,
}

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (config_name, metrics) in zip(axes, configs_to_plot.items()):
    y_pred = (metrics["y_proba"] >= metrics["threshold"]).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred 0", "Pred 1"])
    ax.set_yticklabels(["True 0", "True 1"])
    for row in range(2):
        for col in range(2):
            ax.text(col, row, str(cm[row, col]),
                    ha="center", va="center", fontsize=14, fontweight="bold",
                    color="white" if cm[row, col] > cm.max() / 2 else "black")
    ax.set_title(
        f"{config_name}\nF1={metrics['f1']:.3f}  "
        f"P={metrics['precision']:.3f}  R={metrics['recall']:.3f}",
        fontsize=10, fontweight="bold",
    )
    plt.colorbar(im, ax=ax)

plt.suptitle("Confusion Matrices: Baseline vs Best Strategy",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Visualise resampled distributions (PCA projection) ────────────────────────

pca2 = PCA(n_components=2, random_state=SEED)
pca2.fit(X_train)   # fit on original training data

datasets: dict = {
    "Original (1:19)": (X_train, y_train),
    "Oversampled (1:1)": (X_over, y_over),
    "Undersampled (1:1)": (X_under, y_under),
    "SMOTE (1:1)": (X_smote, y_smote),
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (title, (X_d, y_d)) in zip(axes, datasets.items()):
    X_proj = pca2.transform(X_d)
    n_maj = int((y_d == 0).sum())
    n_min = int((y_d == 1).sum())
    for label, color, name in [(0, "steelblue", "Majority"), (1, "coral", "Minority")]:
        mask = y_d == label
        ax.scatter(X_proj[mask, 0], X_proj[mask, 1],
                   c=color, s=5, alpha=0.4, label=name)
    ax.set_title(f"{title}\n(maj={n_maj:,}, min={n_min:,})",
                 fontsize=9, fontweight="bold")
    ax.set_xlabel("PC1", fontsize=9)
    ax.set_ylabel("PC2", fontsize=9)
    ax.legend(fontsize=7)

plt.suptitle("PCA Projection: Original vs Resampled Datasets",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Cross-validation F1 score comparison (box plots) ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metric_keys = ["f1", "roc_auc", "avg_precision"]
metric_labels = ["F1 (minority class)", "ROC-AUC", "Average Precision"]

config_names = list(cv_results.keys())
colors_cv = ["steelblue", "coral", "green", "purple", "orange"]

for ax, metric_key, metric_label in zip(axes, metric_keys, metric_labels):
    data_for_box = [cv_results[cfg][metric_key] for cfg in config_names]
    bp = ax.boxplot(data_for_box, patch_artist=True, notch=False,
                    medianprops={"color": "black", "linewidth": 2})
    for patch, color in zip(bp["boxes"], colors_cv):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_xticks(range(1, len(config_names) + 1))
    ax.set_xticklabels(config_names, rotation=25, ha="right", fontsize=9)
    ax.set_ylabel(metric_label, fontsize=10)
    ax.set_title(f"{metric_label}\n(5-fold stratified CV)",
                 fontsize=10, fontweight="bold")
    ax.grid(True, axis="y", alpha=0.3)

plt.suptitle("Cross-Validation Metric Distribution by Imbalanced-Data Strategy",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── F_beta analysis: precision vs recall trade-off via beta ───────────────────
# beta < 1 favours precision (e.g., spam filter: avoid false positives)
# beta > 1 favours recall   (e.g., cancer screening: avoid false negatives)

betas = [0.25, 0.5, 1.0, 2.0, 4.0]
y_proba_smote_fb = res_smote["y_proba"]

print("F_beta optimal threshold and score for SMOTE model:")
print(f"{'Beta':>6s}  {'Threshold':>10s}  {'Precision':>10s}  {'Recall':>10s}  {'F_beta':>8s}  {'Interpretation'}")
print("-" * 78)

fbeta_rows = []
for beta in betas:
    thresh, prec, rec, fbeta = find_optimal_threshold(
        y_test, y_proba_smote_fb, beta=beta
    )
    if beta < 1.0:
        interp = "Precision-biased"
    elif beta == 1.0:
        interp = "Balanced"
    else:
        interp = "Recall-biased"
    print(f"{beta:6.2f}  {thresh:10.3f}  {prec:10.3f}  {rec:10.3f}  {fbeta:8.3f}  {interp}")
    fbeta_rows.append((beta, thresh, prec, rec, fbeta))

# Plot precision and recall at optimal threshold for each beta
fig, ax = plt.subplots(figsize=(8, 4))
beta_vals  = [r[0] for r in fbeta_rows]
prec_vals  = [r[2] for r in fbeta_rows]
rec_vals   = [r[3] for r in fbeta_rows]
thresh_vals = [r[1] for r in fbeta_rows]

ax.plot(beta_vals, prec_vals,   marker="o", label="Precision at optimal thresh",
        color="steelblue", linewidth=2)
ax.plot(beta_vals, rec_vals,    marker="s", label="Recall at optimal thresh",
        color="coral", linewidth=2)
ax.plot(beta_vals, thresh_vals, marker="^", label="Threshold",
        color="green", linewidth=2, linestyle="--")
ax.set_xlabel("Beta (F_beta parameter)", fontsize=11)
ax.set_ylabel("Score", fontsize=11)
ax.set_title("Precision-Recall Trade-off via F_beta\n"
             "(higher beta → lower threshold → higher recall, lower precision)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### BalancedBagging Ensemble

A natural extension of undersampling: train **multiple** models, each on a different
random undersample of the majority class, then average their predictions.  This recovers
most of the discarded majority information through ensemble diversity — higher variance per
model is offset by the averaging step.


In [ ]:
# ── BalancedBagging: Ensemble of Undersampled Classifiers ────────────────────
# Each base classifier trains on all minority samples + a random subset of
# majority samples (= size of minority set).  Predictions are averaged.
# This reduces information loss from undersampling without inflating dataset size.


class BalancedBaggingClassifier:
    '''Ensemble that fits each base estimator on a balanced undersampled subset.

    Conceptually similar to sklearn's BalancedBaggingClassifier.  For each of
    n_estimators base classifiers, draws all minority samples and a fresh
    random subset of majority samples of equal size.

    Attributes:
        n_estimators: Number of base classifiers.
        base_estimator_class: Class of the base classifier (must support fit /
            predict_proba).
        estimator_kwargs: Keyword arguments forwarded to the base classifier.
        seed: Master random seed.
        estimators_: List of fitted estimator instances (populated after fit).
    '''

    def __init__(
        self,
        n_estimators: int = 10,
        base_estimator_class: Any = None,
        seed: int = SEED,
        **estimator_kwargs: Any,
    ) -> None:
        '''Store configuration; defer fitting to fit().

        Args:
            n_estimators: Number of independent base classifiers to train.
            base_estimator_class: Classifier class (default: LogisticRegression).
            seed: Master random seed; each estimator gets seed + estimator_index.
            **estimator_kwargs: Extra kwargs forwarded to base_estimator_class.
        '''
        if base_estimator_class is None:
            base_estimator_class = LogisticRegression
        self.n_estimators        = n_estimators
        self.base_estimator_class = base_estimator_class
        self.estimator_kwargs    = estimator_kwargs
        self.seed                = seed
        self.estimators_: list   = []

    def fit(self, X: np.ndarray, y: np.ndarray) -> "BalancedBaggingClassifier":
        '''Train n_estimators classifiers, each on a fresh balanced subset.

        Args:
            X: Feature matrix of shape (n_samples, n_features).
            y: Binary label vector of shape (n_samples,).

        Returns:
            self
        '''
        self.estimators_ = []
        minority_idx = np.where(y == 1)[0]
        majority_idx = np.where(y == 0)[0]
        n_minority   = len(minority_idx)

        for i in range(self.n_estimators):
            rng = np.random.default_rng(self.seed + i)
            maj_sample = rng.choice(majority_idx, size=n_minority, replace=False)
            idx = np.concatenate([minority_idx, maj_sample])
            rng.shuffle(idx)
            X_bal, y_bal = X[idx], y[idx]

            est = self.base_estimator_class(
                random_state=self.seed + i, **self.estimator_kwargs
            )
            est.fit(X_bal, y_bal)
            self.estimators_.append(est)

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        '''Average predicted probabilities across all base estimators.

        Args:
            X: Feature matrix of shape (n_samples, n_features).

        Returns:
            Probability array of shape (n_samples, n_classes).
        '''
        proba_sum = np.zeros((X.shape[0], 2), dtype=np.float64)
        for est in self.estimators_:
            proba_sum += est.predict_proba(X)
        return proba_sum / self.n_estimators

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        '''Predict class labels using averaged probabilities.

        Args:
            X: Feature matrix of shape (n_samples, n_features).
            threshold: Decision threshold for the positive class.

        Returns:
            Integer label array of shape (n_samples,).
        '''
        proba = self.predict_proba(X)[:, 1]
        return (proba >= threshold).astype(int)


# ── Train BalancedBagging and compare to single-model baseline ────────────────

bbc = BalancedBaggingClassifier(
    n_estimators=10,
    base_estimator_class=LogisticRegression,
    seed=SEED,
    max_iter=1000,
)
bbc.fit(X_train, y_train)

y_proba_bbc = bbc.predict_proba(X_test)[:, 1]
y_pred_bbc  = bbc.predict(X_test)

tp_b = int(((y_pred_bbc == 1) & (y_test == 1)).sum())
fp_b = int(((y_pred_bbc == 1) & (y_test == 0)).sum())
fn_b = int(((y_pred_bbc == 0) & (y_test == 1)).sum())
prec_b = tp_b / (tp_b + fp_b) if (tp_b + fp_b) > 0 else 0.0
rec_b  = tp_b / (tp_b + fn_b) if (tp_b + fn_b) > 0 else 0.0
f1_b   = 2 * prec_b * rec_b / (prec_b + rec_b) if (prec_b + rec_b) > 0 else 0.0
ap_b   = average_precision_score(y_test, y_proba_bbc)

print("BalancedBagging (10 estimators) vs single-model strategies:")
print(f"  Precision      : {prec_b:.3f}")
print(f"  Recall         : {rec_b:.3f}")
print(f"  F1             : {f1_b:.3f}")
print(f"  Avg Precision  : {ap_b:.3f}")
print(f"  (Baseline F1   : {res_baseline['f1']:.3f}  SMOTE F1: {res_smote['f1']:.3f})")

# Add BBC to all_results for final comparison
all_results["BalancedBagging"] = {
    "threshold"     : 0.5,
    "precision"     : prec_b,
    "recall"        : rec_b,
    "f1"            : f1_b,
    "roc_auc"       : roc_auc_score(y_test, y_proba_bbc),
    "avg_precision" : ap_b,
    "y_proba"       : y_proba_bbc,
}
print("BalancedBagging evaluated and added to results.")


### Imbalance Ratio Sensitivity Analysis

How robust is each strategy as the dataset becomes more severely imbalanced?  We sweep
minority fractions from 20% (mild) down to 1% (severe), training Logistic Regression
under each strategy and measuring minority-class F1 on a held-out test set.


In [ ]:
# ── Imbalance ratio sensitivity: F1 vs minority fraction ─────────────────────
# As the imbalance ratio worsens, how quickly does each strategy degrade?
# We sweep minority fractions from 20% down to 1% and measure test F1.


def evaluate_at_imbalance(
    minority_fraction: float,
    n_samples: int = 3000,
    seed: int = SEED,
) -> dict[str, float]:
    '''Generate an imbalanced dataset and evaluate four strategies.

    Generates a fresh classification problem at the specified minority fraction,
    trains Logistic Regression under each strategy, and returns test F1 scores.

    Args:
        minority_fraction: Fraction of positive-class samples (0 < f < 0.5).
        n_samples: Total sample count for the generated dataset.
        seed: Random seed.

    Returns:
        Dict mapping strategy name to test F1 score.
    '''

    X_d, y_d = make_classification(
        n_samples=n_samples,
        n_features=15,
        n_informative=6,
        n_redundant=3,
        weights=[1.0 - minority_fraction, minority_fraction],
        flip_y=0.01,
        random_state=seed,
    )
    X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
        X_d, y_d, test_size=0.2, stratify=y_d, random_state=seed
    )
    sc = StandardScaler()
    X_tr_d = sc.fit_transform(X_tr_d)
    X_te_d = sc.transform(X_te_d)

    # Skip if minority class has fewer than k+1 samples for SMOTE
    if (y_tr_d == 1).sum() < 6:
        return {}

    f1_scores_local: dict[str, float] = {}

    # Baseline
    m = LogisticRegression(max_iter=500, random_state=seed)
    m.fit(X_tr_d, y_tr_d)
    f1_scores_local["Baseline"] = f1_score(y_te_d, m.predict(X_te_d), zero_division=0)

    # Class-weighted
    cw_d = compute_class_weights(y_tr_d, strategy="inverse_frequency")
    sw_d = np.array([cw_d[int(c)] for c in y_tr_d])
    m2 = LogisticRegression(max_iter=500, random_state=seed)
    m2.fit(X_tr_d, y_tr_d, sample_weight=sw_d)
    f1_scores_local["Class-weighted"] = f1_score(y_te_d, m2.predict(X_te_d), zero_division=0)

    # SMOTE
    X_sm, y_sm = smote_oversample(X_tr_d, y_tr_d, target_ratio=1.0,
                                   k_neighbours=5, seed=seed)
    m3 = LogisticRegression(max_iter=500, random_state=seed)
    m3.fit(X_sm, y_sm)
    f1_scores_local["SMOTE"] = f1_score(y_te_d, m3.predict(X_te_d), zero_division=0)

    # Undersampled
    X_us, y_us = random_undersample(X_tr_d, y_tr_d, target_ratio=1.0, seed=seed)
    m4 = LogisticRegression(max_iter=500, random_state=seed)
    m4.fit(X_us, y_us)
    f1_scores_local["Undersampled"] = f1_score(y_te_d, m4.predict(X_te_d), zero_division=0)

    return f1_scores_local


minority_fracs = [0.20, 0.15, 0.10, 0.08, 0.05, 0.03, 0.02, 0.01]
sweep_results: dict[str, list] = {
    "Baseline": [], "Class-weighted": [], "SMOTE": [], "Undersampled": []
}
valid_fracs: list = []

print("Imbalance ratio sensitivity sweep:")
print(f"{'Min frac':>9s}  {'Ratio':>6s}  {'Baseline':>9s}  {'Weighted':>9s}  {'SMOTE':>7s}  {'Undersamp':>10s}")
print("-" * 60)

for frac in minority_fracs:
    res_d = evaluate_at_imbalance(frac, n_samples=3000, seed=SEED)
    if not res_d:
        continue
    valid_fracs.append(frac)
    ratio = (1.0 - frac) / frac
    for strat in sweep_results:
        sweep_results[strat].append(res_d.get(strat, float("nan")))
    print(
        f"{frac:9.2f}  {ratio:6.1f}  "
        f"{res_d['Baseline']:9.3f}  "
        f"{res_d['Class-weighted']:9.3f}  "
        f"{res_d['SMOTE']:7.3f}  "
        f"{res_d['Undersampled']:10.3f}"
    )

fig, ax = plt.subplots(figsize=(9, 5))
sweep_colors = ["steelblue", "coral", "green", "purple"]
for (strat, f1_list), color in zip(sweep_results.items(), sweep_colors):
    ax.plot(valid_fracs, f1_list, marker="o", label=strat, color=color, linewidth=2)

ax.invert_xaxis()   # worsening imbalance goes left
ax.set_xlabel("Minority class fraction (decreasing → more imbalanced)", fontsize=11)
ax.set_ylabel("Test F1 (minority class)", fontsize=11)
ax.set_title("Strategy Robustness vs Imbalance Severity",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("Imbalance ratio sensitivity analysis complete.")


---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Accuracy is misleading on imbalanced data.** A model that always predicts the majority class
   achieves 95% accuracy on a 1:19 dataset while detecting **zero** minority examples.  Always
   report F1, average precision, and PR-AUC for imbalanced problems.

2. **PR curves are preferred over ROC for imbalanced data.** ROC-AUC is dominated by the large
   number of true negatives (majority class), making even poor minority detectors look good.
   PR curves ignore true negatives entirely and directly capture minority-class detection quality.

3. **Resampling must happen inside each CV fold.** Applying SMOTE or oversampling before splitting
   leaks synthetic minority data into the validation fold — the "same" (interpolated) examples may
   appear in both train and val, producing optimistically biased estimates.

4. **SMOTE generalises better than random oversampling** because it generates novel examples along
   the minority-class manifold rather than duplicating existing points, reducing the risk of
   overfitting to specific minority examples.

5. **Threshold tuning is cheap and often surprisingly effective.** After choosing a resampling
   strategy, optimising the decision threshold (using F_beta on a validation set) frequently
   yields further F1 gains with no additional training cost.  Set beta based on domain requirements:
   beta > 1 for high-recall applications (cancer screening), beta < 1 for high-precision (spam).

### What's Next

→ **04-06 (Learning Theory — VC Dimension & PAC Learning)** provides the theoretical framework
that explains *why* models generalise — formalising the intuition that model capacity must match
data quantity to avoid the high-variance regime we diagnosed via PR curves here.